# F6-svd-spectral — Practice p25 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** gram-matrices, spectral-decomposition, low-rank-approximation, frobenius-norm

*Coding and reasoning are required. Parts (a)–(f) form one chain; each part must consume the named results from the preceding parts.*

A seeded integer feature matrix $W\in\mathbb{R}^{9\times4}$ is supplied below; you may take as **given** that $W$ has full column rank (rank 4). Work through its row-Gram matrix $S=WW^{\mathsf T}$ without replacing the required routes.

**(a) Gram matrix.** Build `S` from `W` with broadcasting and one axis sum. Set `symmetry_ok` and `rank_ceiling_ok` to booleans asserting, respectively, that $S$ is symmetric and that $\operatorname{rank}(S)\le4$. **Banned in part (a) (zero points): `@`, `np.matmul`, and `np.dot`; wrapping one in a helper is still banned.**

**(b) Spectrum.** Call `np.linalg.eigh(S)`, then apply the pinned `[::-1]` reorder to both eigenvalues and eigenvector columns. Store the results as `lam_desc` (shape `(9,)`) and `Q_desc` (shape `(9, 9)`). Reconstruct $Q\Lambda Q^{\mathsf T}$ as `S_reconstructed`, compute `reconstruction_gap` as its Frobenius-distance from `S`, and assert with `np.isclose(..., atol=1e-9, rtol=0)` that the gap is zero to tolerance.

**(c) Mirror bridge.** Obtain `sigma` from `np.linalg.svd(W, compute_uv=False)`. Set `bridge_gap` to the largest absolute difference between `lam_desc[:4]` and `sigma**2`. The comparison is over the four nonzero eigenvalues only. Handle the remaining five eigenvalues by the invariant `zero_block_energy = sum(lam_desc[4:]**2)` and assert both quantities are zero to `atol=1e-9, rtol=0`. Do not compare eigenvector columns.

**(d) Every truncation from one tail pass.** For $r=1,2,3,4$, the spectral tail identity gives $\lVert S-S_r\rVert_F^2=\sum_{i>r}\lambda_i^2$. Use exactly one `np.cumsum` on the reversed squared spectrum and store the four relative squared errors, in rank order, as `rel_err2` (shape `(4,)`). **Banned in part (d) (zero points): constructing any `S_r` explicitly, taking four separate sums, or calling `np.cumsum` more than once.**

**(e) The ceiling, not a budget.** The Gram construction forces a hard limit: no rank-$r$ approximation with $r>4$ can improve on $r=4$. Set `zero_beyond_ceiling` to the relative squared error at $r=4$ and assert it is zero to `atol=1e-9, rtol=0`. The scaffold asks you to commit `predicted_zero_count` — the number of zero eigenvalues you predict — at the top of the cell, before `lam_desc` exists. Argue that prediction from the shape of $W$ together with the given full column rank, and note in one sentence that the shape alone gives only a *lower bound* of five: full column rank is what makes the count exact. Then set `observed_zero_count`, counted from `lam_desc` with the same tolerance, and assert that the two agree. **Banned in part (e) (zero points): reading `lam_desc` to obtain `predicted_zero_count`; it must follow from the shape argument.**

**(f) Written conclusion.** In 3–5 sentences, explain why forming a $9\times9$ Gram matrix from a $9\times4$ factor cannot create more than four nonzero spectral directions, and why that ceiling — not any error budget — is what bounds the achievable approximation here. State what the five zero eigenvalues mean geometrically for the rows of $W$.


In [1]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
W = rng.integers(-5, 6, size=(9, 4))

# Shape-only prediction, committed before lam_desc exists.
predicted_zero_count = W.shape[0] - W.shape[1]

# (a) Each pair of rows is multiplied coordinatewise, then reduced once.
S = (W[:, None, :] * W[None, :, :]).sum(axis=2)
symmetry_ok = bool(np.allclose(S, S.T, atol=1e-12, rtol=0))
rank_ceiling_ok = bool(np.linalg.matrix_rank(S) <= W.shape[1])

# (b) eigh is ascending, so the same reversal must act on values and columns.
lam_asc, Q_asc = np.linalg.eigh(S)
lam_desc = lam_asc[::-1]
Q_desc = Q_asc[:, ::-1]
S_reconstructed = (Q_desc * lam_desc) @ Q_desc.T
reconstruction_gap = np.linalg.norm(S_reconstructed - S, ord="fro")
assert np.isclose(reconstruction_gap, 0.0, atol=1e-9, rtol=0)

# (c) Compare spectra, not eigenvectors inside the degenerate zero block.
sigma = np.linalg.svd(W, compute_uv=False)
bridge_gap = np.max(np.abs(lam_desc[:4] - sigma**2))
zero_block_energy = np.sum(lam_desc[4:]**2)
assert np.isclose(bridge_gap, 0.0, atol=1e-9, rtol=0)
assert np.isclose(zero_block_energy, 0.0, atol=1e-9, rtol=0)

# (d) One reverse cumulative pass contains every needed tail sum.
tail_sq = np.cumsum((lam_desc**2)[::-1])[::-1]
rel_err2 = tail_sq[1:5] / tail_sq[0]

# (e) Compare the earlier shape prediction with the observed spectrum.
zero_beyond_ceiling = rel_err2[3]
observed_zero_count = int(
    np.isclose(lam_desc, 0.0, atol=1e-9, rtol=0).sum()
)
assert np.isclose(zero_beyond_ceiling, 0.0, atol=1e-9, rtol=0)
assert predicted_zero_count == observed_zero_count

print("W:\n", W)
print("eigenvalues (descending):", lam_desc)
print("relative squared errors r=1..4:", rel_err2)
print("predicted / observed zeros:", predicted_zero_count, observed_zero_count)


W:
 [[ 3 -3  1 -4]
 [ 2  1 -1 -3]
 [-3  5  3  5]
 [ 3  3  4 -5]
 [ 4 -2 -1 -4]
 [-1  4  0 -3]
 [-5 -5 -2 -5]
 [ 1  5  4  4]
 [ 2 -5  1  3]]
eigenvalues (descending): [ 2.10039020e+02  1.14386955e+02  7.16165861e+01  1.99574394e+01
  1.74048487e-14  1.14797005e-14  3.59256778e-15  2.15159005e-15
 -1.48859275e-14]
relative squared errors r=1..4: [2.96703390e-01 8.81143156e-02 6.34962679e-03 1.07422363e-32]
predicted / observed zeros: 5 5


The map $W^{\mathsf T}$ sends vectors from $\mathbb{R}^9$ into only $\mathbb{R}^4$, so $S=WW^{\mathsf T}$ cannot have more than four nonzero spectral directions. Here the seeded $W$ has full column rank, so the shape ceiling is attained and the other five eigenvalues are zero. Consequently the rank-4 truncation already equals $S$ to numerical tolerance, and increasing $r$ cannot improve it; this is a rank ceiling, not a choice made from an error budget. Geometrically, the five zero-eigenvalue directions form the null space orthogonal to the four-dimensional span of the columns of $W$ (equivalently, five independent coefficient directions among the nine rows collapse under $W^{\mathsf T}$).

### Answer check

In [2]:
assert W.shape == (9, 4) and S.shape == (9, 9)
assert symmetry_ok is True and rank_ceiling_ok is True
assert lam_desc.shape == (9,) and Q_desc.shape == (9, 9)
assert np.isclose(reconstruction_gap, 0.0, atol=1e-9, rtol=0)
assert np.isclose(bridge_gap, 0.0, atol=1e-9, rtol=0)
assert np.isclose(zero_block_energy, 0.0, atol=1e-9, rtol=0)
assert rel_err2.shape == (4,)
assert np.isclose(zero_beyond_ceiling, 0.0, atol=1e-9, rtol=0)
assert predicted_zero_count == observed_zero_count == 5